# Fine-tuning YOLOv11 Segmentation — Détection de Fissures

Ce notebook exécute l'entraînement complet de YOLOv11 pour la segmentation d'instances de fissures.

## 1. Montage de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration des chemins
Chaque dataset est une variable de chemin **indépendante** : collez le chemin réel
de chaque dossier partagé de votre Drive (peu importe où il se trouve).

In [ ]:
import os, shutil

# ─── À adapter ────────────────────────────────────────────
DRIVE_PROJECT = '/content/drive/MyDrive/Projet_Fissures'   # Dossier du projet sur Drive

# Dataset YOLOv11 : dossier avec data.yaml + train/valid/test
DATASET_YOLO = '/content/drive/MyDrive/segmentation_fissures.v6i.yolov11'

# Murs sains (optionnel) : dossier d'images sans fissure
MURS_SAINS = '/content/drive/MyDrive/murs_sains'

# Si le raccourci ci-dessus est "cassé" (ls renvoie No such file or directory
# alors que le dossier apparaît dans MyDrive avec des droits 'l...'), utiliser
# plutôt les chemins directs par ID de raccourci (voir README, section
# "raccourci Drive cassé") :
# DATASET_YOLO = '/content/drive/.shortcut-targets-by-id/1XbQnRhiAGmmrZf3il4ce9NJNcFMiGsBM/segmentation_fissures.v6i.yolov11'
# MURS_SAINS   = '/content/drive/.shortcut-targets-by-id/1Kan2sZwrdw_F-ZrYm7_tAwxBfE6s2hiL/murs_sains'
# ──────────────────────────────────────────────────────────

PROJECT_DIR = '/content/Projet'

# Copie du projet
shutil.copytree(DRIVE_PROJECT, PROJECT_DIR, dirs_exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'Projet chargé dans : {PROJECT_DIR}')

# Vérification des chemins de dataset
for path, label in [(DATASET_YOLO, 'Dataset YOLO'), (MURS_SAINS, 'Murs sains')]:
    if os.path.isdir(path):
        print(f'  [OK] {label} : {path}  ->  {os.listdir(path)[:5]}')
    else:
        print(f'  [INTROUVABLE] {label} : {path}')

## 3. Installation des dépendances

In [ ]:
!pip install -q ultralytics opencv-python-headless pycocotools pyyaml tqdm

## 4. Vérification du GPU

In [ ]:
import torch
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA dispo  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 5. Entraînement YOLOv11

In [ ]:
# Entraînement depuis zéro
!python scripts/train_yolov11.py \
    --data-root {DATASET_YOLO} \
    --murs-sains {MURS_SAINS} \
    --config configs/yolov11_config.yaml \
    --device 0

In [ ]:
# ─── Cellule optionnelle : surcharger les hyperparamètres ───
# Décommenter et adapter selon vos besoins

# !python scripts/train_yolov11.py \
#     --data-root {DATASET_YOLO} \
#     --murs-sains {MURS_SAINS} \
#     --config configs/yolov11_config.yaml \
#     --epochs 150 \
#     --batch 16 \
#     --imgsz 1280 \
#     --device 0 \
#     --name run_hires

## 6. Reprendre un entraînement interrompu

In [ ]:
# Décommenter pour reprendre depuis le dernier checkpoint
# !python scripts/train_yolov11.py \
#     --data-root {DATASET_YOLO} \
#     --resume outputs/yolov11/run/weights/last.pt

## 7. Visualisation des résultats

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

run_dir = Path('outputs/yolov11/run')
plots = list(run_dir.glob('*.png'))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, plot in enumerate(plots[:6]):
    img = mpimg.imread(str(plot))
    axes[i].imshow(img)
    axes[i].set_title(plot.stem, fontsize=10)
    axes[i].axis('off')

for j in range(len(plots), 6):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Afficher les métriques finales
import pandas as pd
results_csv = run_dir / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print('\nMeilleures métriques :')
    best_idx = df['metrics/mAP50(M)'].idxmax() if 'metrics/mAP50(M)' in df.columns else df.index[-1]
    print(df.iloc[best_idx].to_string())
    df[['epoch', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)',
        'metrics/precision(M)', 'metrics/recall(M)']].plot(
        x='epoch', figsize=(12, 5), title='Métriques Masque par époque')
    plt.tight_layout()
    plt.show()

## 8. Sauvegarde des résultats vers Drive

In [ ]:
import shutil
dest = f'{DRIVE_PROJECT}/outputs/yolov11'
shutil.copytree('outputs/yolov11', dest, dirs_exist_ok=True)
print(f'Résultats sauvegardés vers : {dest}')